## Metrics Explanation  

This section provides a detailed explanation of the metrics calculated in this notebook for analyzing movement patterns based on tracking data grouped by **patients (`ID`)** and sperm cells (`tracker_id`).  

### 1. **VCL (Curvilinear Velocity)**  
- **Definition**: VCL represents the average velocity of a tracker along its actual curvilinear path. It measures the distance traveled over time, calculated for each sperm within a patient's dataset.  
- **Mathematical Formula**:  

$$v_{ci} = \frac{\|p_{i+1} - p_i\| + \|p_i - p_{i-1}\|}{2\Delta t}$$
$$\text{VCL} = \frac{1}{N-2} \sum_{i=1}^{N-1} v_{ci}$$

  Where:  
  - $p_i$: Position at frame $i$ (e.g., $(x_{center}, y_{center})$).  
  - $\|p_{i+1} - p_i\|$: Euclidean distance between positions at frames \(i+1\) and \(i\).  
  - $\|p_i - p_{i-1}\|$: Euclidean distance between positions at frames \(i\) and \(i-1\).  
  - $\Delta t$: Time interval between frames.  
  - $N$: Total number of frames for the tracker.

### 2. **VSL (Straight-Line Velocity)**  
- **Definition**: VSL measures the average velocity along a straight line connecting the initial and final positions of a sperm cell. It quantifies the efficiency of movement in terms of directness.  
- **Mathematical Formula**:  
  $$\text{VSL} = \frac{\|p_{\text{end}} - p_{\text{start}}\|}{T}$$  
  Where:  
  - $p_{\text{start}}$: Starting position.  
  - $p_{\text{end}}$: Ending position.  
  - $T$: Total time ($N \cdot \Delta t$).  

### 3. **VAP (Average Path Velocity)**  
- **Definition**: VAP represents the average velocity along the path taken by the sperm cell, providing a measure of its average speed over time.  
- **Mathematical Formula**:  
  $$\text{total\_distance} = \sum_{i=1}^{n-1} \sqrt{(x_i - x_{i-1})^2 + (y_i - y_{i-1})^2}$$  
  $$\text{total\_time} = (n - 1) \times \Delta t$$  
  $$\text{VAP} = \frac{\text{total\_distance}}{\text{total\_time}}$$  
  Where:  
  - $x_i$, $y_i$: Coordinates of the position at frame $i$.  
  - $n$: Total number of frames for the tracker.  
  - $\Delta t$: Time interval between frames.  

### 4. **ALH (Amplitude of Lateral Head Displacement)**  
- **Definition**: ALH quantifies the average deviation of the tracker’s position from its average path. It reflects the lateral movement relative to the trajectory.  
- **Mathematical Formula**:  
  $$\text{ALH} = \frac{1}{N} \sum_{i=1}^{N} \|\bar{p} - p_i\|$$  
  Where:  
  - $\bar{p}$: Mean position of the tracker over all frames (average path center).  
  - $p_i$: Position at frame $i$.  

### 5. **MAD (Mean Angular Displacement)**  
- **Definition**: MAD measures the average angular displacement between three consecutive positions. It is useful for analyzing the directional changes in movement.  
- **Mathematical Formula**:  
  $$\theta_i = \cos^{-1}\left(\frac{(p_i - p_{i-1}) \cdot (p_{i+1} - p_i)}{\|p_i - p_{i-1}\| \cdot \|p_{i+1} - p_i\|}\right)$$  
  $$\text{MAD} = \frac{1}{N-2} \sum_{i=2}^{N-1} |\theta_i|$$  
  Where:  
  - $\theta_i$: Angle between vectors formed by three consecutive points.  
  - $\cdot$: Dot product of two vectors.  
  - $\| \cdot \|$: Magnitude of a vector.  

---

### Notes  
- Metrics are computed for each tracker ID grouped by patient ID.  
- $N$: Total number of frames per sperm tracker.  
- All calculations assume the data is ordered temporally (e.g., by `ID`).  
- Metrics such as VCL, VSL, and VAP are expressed in units of distance per time (e.g., $\mu m/s$), while ALH is expressed in units of distance (e.g., $\mu m$), and MAD is in degrees.

# Imports

In [6]:
import pandas as pd
from nb_utils import set_root
import numpy as np

PROJECT_DIR = set_root(2)

# Parameters

In [7]:
path_data = PROJECT_DIR / "data"
path_intermediate = path_data / "02_intermediate"
path_primary = path_data / "03_primary"

file_path_data = path_primary / "tracker.parquet"
file_path_horm = path_intermediate / "data_horm_concat.parquet"
tracker_columns = ["tracker_id",	"class_id",	"x_min",	"y_min",	"x_max",	"y_max",	"x_center",	"y_center"]

# Data

In [15]:
data = pd.read_parquet(file_path_data)
data_horm = pd.read_parquet(file_path_horm)
data

,tracker_id,class_id,x_min,y_min,x_max,y_max,ID,x_center,y_center
0,0,0,4.348797,118.514236,22.273115,138.625229,47,13.310956,128.569733
1,1,0,485.946716,145.540283,506.476501,166.668884,47,496.211609,156.104584
2,2,0,28.142849,251.524780,49.713924,272.921570,47,38.928387,262.223175
3,3,0,103.487030,363.432861,124.152924,384.810669,47,113.819977,374.121765
4,4,0,81.397095,360.035797,98.433258,378.959076,47,89.915176,369.497437
...,...,...,...,...,...,...,...,...,...
740013,4,0,223.239960,271.837341,243.945984,292.405579,22,233.592972,282.121460
740014,218,0,3.530352,246.088257,22.795349,268.874695,22,13.162850,257.481476
740015,174,0,39.091930,24.299667,56.279087,38.885193,22,47.685509,31.592430
740016,211,0,227.979538,456.617737,247.808365,477.442383,22,237.893951,467.030060


# Functions

In [9]:
def calculate_metrics(df):
    """
    Calculates the VCL, VSL, VAP, ALH, and MAD metrics for each tracker_id grouped by patient ID.

    Args:
        df (pd.DataFrame): DataFrame with the columns ['ID', 'tracker_id', 'x_center', 'y_center'].

    Returns:
        pd.DataFrame: DataFrame containing the metrics per tracker_id grouped by patient ID.
    """
    results = []
    
    for patient_id, patient_group in df.groupby('ID'):
        
        for tracker_id, group in patient_group.groupby('tracker_id'):
            group = group.sort_values('ID')
            positions = group[['x_center', 'y_center']].values
            delta_t = 0.01  # Ajustar (representa o número de frames consecutivos)

            # VCL
            vcl = np.mean([
                (np.linalg.norm(positions[i] - positions[i - 1]) + np.linalg.norm(positions[i + 1] - positions[i])) / (2 * delta_t)
                for i in range(1, len(positions) - 1)
            ]) if len(positions) > 2 else 0

            # VSL
            vsl = (
                np.linalg.norm(positions[-1] - positions[0]) / (len(positions) * delta_t)
                if len(positions) > 1 else 0
            )

            # VAP
            total_distance = sum(np.linalg.norm(positions[i] - positions[i - 1]) for i in range(1, len(positions)))
            total_time = (len(positions) - 1) * delta_t
            vap = total_distance / total_time if total_time > 0 else 0

            # ALH
            avg_path = np.mean(positions, axis=0)
            alh = np.mean([
                np.linalg.norm(positions[i] - avg_path)
                for i in range(len(positions))
            ]) if len(positions) > 1 else 0

            # MAD
            mad = np.mean([
                np.abs(np.degrees(np.arccos(
                    np.clip(
                        np.dot(positions[i] - positions[i - 1], positions[i + 1] - positions[i]) /
                        (np.linalg.norm(positions[i] - positions[i - 1]) *
                         np.linalg.norm(positions[i + 1] - positions[i])),
                        -1.0, 1.0
                    )
                )))
                for i in range(1, len(positions) - 1)
            ]) if len(positions) > 2 else 0

            results.append({
                'ID': patient_id,
                'tracker_id': tracker_id,
                'VCL': vcl,
                'VSL': vsl,
                'VAP': vap,
                'ALH': alh,
                'MAD': mad
            })

    metrics_df = pd.DataFrame(results)
    return metrics_df

In [10]:
metrics_df = calculate_metrics(data)
metrics_df

,ID,tracker_id,VCL,VSL,VAP,ALH,MAD
0,11,0,54.272826,4.263747,57.012226,12.385463,95.483721
1,11,1,552.063376,95.182066,618.490487,11.002528,42.374299
2,11,2,46.915004,3.856542,49.495935,5.312231,99.865077
3,11,3,47.857591,1.803621,48.447662,1.156313,105.151219
4,11,4,18.255177,0.325854,18.468413,0.615399,98.851131
...,...,...,...,...,...,...,...
11001,82,367,681.637259,156.491119,767.219953,17.935375,90.792764
11002,82,368,96.183178,20.919680,102.761531,0.848048,101.616238
11003,82,369,86.415946,5.595145,93.892855,1.407855,124.515828
11004,82,370,19.172055,4.213669,19.116416,0.214091,45.727466
